# Part 4 — Part 4 — LLM-Powered Feature: Structured Extraction, Tabular Batch Scoring, or Model Prediction Explanation

## Track Selected

> I selected **Track B - Tabular Record Batch Scoring**.

- In this part, I used an LLM API to check house records from my cleaned dataset.
The LLM takes each house record as JSON input and returns a structured JSON assessment.

- I also added JSON validation and a simple PII safety check before sending data to the LLM.

In [74]:
# Installing Required Libraries
!pip install requests jsonschema python-dotenv

## Importing Required Libraries

- Here I imported all libraries that I need for loading data, sending API requests, checking JSON format and applying safety checks.

In [75]:
import pandas as pd
import numpy as np

import requests
import json
import os
import re

from jsonschema import validate, ValidationError

## Loading Dataset...

- I used the cleaned_data.csv file from Part 1.

- From this dataset, I selected three house records and converted them into JSON format because the LLM will process structured data.

In [76]:
# Dataset...
df = pd.read_csv("cleaned_data.csv")
print("Dataset shape:", df.shape)

# Selecting three random house records
records = df.sample(3, random_state=42)
records

# converting dataframe rows into JSON objects
json_records = []
for _, row in records.iterrows():
    json_records.append(
        row.to_dict()
    )
print(json.dumps(json_records[0], indent=2))

Dataset shape: (1460, 81)
{
  "Id": 893,
  "MSSubClass": 20,
  "MSZoning": "RL",
  "LotFrontage": 70.0,
  "LotArea": 8414,
  "Street": "Pave",
  "Alley": NaN,
  "LotShape": "Reg",
  "LandContour": "Lvl",
  "Utilities": "AllPub",
  "LotConfig": "Inside",
  "LandSlope": "Gtl",
  "Neighborhood": "Sawyer",
  "Condition1": "Norm",
  "Condition2": "Norm",
  "BldgType": "1Fam",
  "HouseStyle": "1Story",
  "OverallQual": 6,
  "OverallCond": 8,
  "YearBuilt": 1963,
  "YearRemodAdd": 2003,
  "RoofStyle": "Hip",
  "RoofMatl": "CompShg",
  "Exterior1st": "HdBoard",
  "Exterior2nd": "HdBoard",
  "MasVnrType": NaN,
  "MasVnrArea": 0.0,
  "ExterQual": "TA",
  "ExterCond": "TA",
  "Foundation": "CBlock",
  "BsmtQual": "TA",
  "BsmtCond": "TA",
  "BsmtExposure": "No",
  "BsmtFinType1": "GLQ",
  "BsmtFinSF1": 663,
  "BsmtFinType2": "Unf",
  "BsmtFinSF2": 0,
  "BsmtUnfSF": 396,
  "TotalBsmtSF": 1059,
  "Heating": "GasA",
  "HeatingQC": "TA",
  "CentralAir": "Y",
  "Electrical": "SBrkr",
  "1stFlrSF": 106

## Converting House Records into Clean JSON Format

In this step, I am selecting a few house records from my dataset and converting them into JSON format.

- JSON format is useful because LLM applications usually work with structured data. Before sending data to an LLM, I need to make sure the data is clean and follows proper JSON rules.

- I am also replacing missing values with `null` because JSON does not support `NaN`.

In [77]:
# Selecting three random house records
records = df.sample(3, random_state=42)

# Replacing missing values with None for valid JSON conversion
records = records.where(pd.notnull(records), None)

# Converting dataframe rows into JSON objects
json_records = []

for _, row in records.iterrows():
    json_records.append(row.to_dict())

# Displaying first JSON record
print(json.dumps(json_records[0], indent=2))

{
  "Id": 893,
  "MSSubClass": 20,
  "MSZoning": "RL",
  "LotFrontage": 70.0,
  "LotArea": 8414,
  "Street": "Pave",
  "Alley": null,
  "LotShape": "Reg",
  "LandContour": "Lvl",
  "Utilities": "AllPub",
  "LotConfig": "Inside",
  "LandSlope": "Gtl",
  "Neighborhood": "Sawyer",
  "Condition1": "Norm",
  "Condition2": "Norm",
  "BldgType": "1Fam",
  "HouseStyle": "1Story",
  "OverallQual": 6,
  "OverallCond": 8,
  "YearBuilt": 1963,
  "YearRemodAdd": 2003,
  "RoofStyle": "Hip",
  "RoofMatl": "CompShg",
  "Exterior1st": "HdBoard",
  "Exterior2nd": "HdBoard",
  "MasVnrType": null,
  "MasVnrArea": 0.0,
  "ExterQual": "TA",
  "ExterCond": "TA",
  "Foundation": "CBlock",
  "BsmtQual": "TA",
  "BsmtCond": "TA",
  "BsmtExposure": "No",
  "BsmtFinType1": "GLQ",
  "BsmtFinSF1": 663,
  "BsmtFinType2": "Unf",
  "BsmtFinSF2": 0,
  "BsmtUnfSF": 396,
  "TotalBsmtSF": 1059,
  "Heating": "GasA",
  "HeatingQC": "TA",
  "CentralAir": "Y",
  "Electrical": "SBrkr",
  "1stFlrSF": 1068,
  "2ndFlrSF": 0,
  "L

## Creating Prompt for LLM

Now I am creating a prompt using the house JSON data.

- The purpose of this prompt is to give structured house information to an LLM and ask it to generate a simple explanation of the property.

- This is similar to how real-world AI applications provide data to an LLM and get meaningful responses.

In [78]:
# Taking one house record as input for the prompt
house_data = json_records[0]

# Creating prompt for LLM
prompt = f"""
You are a real estate assistant.

Analyze the following house details and provide a simple description.

House Details:
{json.dumps(house_data, indent=2)}

Explain:
1. Overall quality of the house
2. Important features
3. Possible buyer advantages
4. A short summary
"""

print(prompt)


You are a real estate assistant.

Analyze the following house details and provide a simple description.

House Details:
{
  "Id": 893,
  "MSSubClass": 20,
  "MSZoning": "RL",
  "LotFrontage": 70.0,
  "LotArea": 8414,
  "Street": "Pave",
  "Alley": null,
  "LotShape": "Reg",
  "LandContour": "Lvl",
  "Utilities": "AllPub",
  "LotConfig": "Inside",
  "LandSlope": "Gtl",
  "Neighborhood": "Sawyer",
  "Condition1": "Norm",
  "Condition2": "Norm",
  "BldgType": "1Fam",
  "HouseStyle": "1Story",
  "OverallQual": 6,
  "OverallCond": 8,
  "YearBuilt": 1963,
  "YearRemodAdd": 2003,
  "RoofStyle": "Hip",
  "RoofMatl": "CompShg",
  "Exterior1st": "HdBoard",
  "Exterior2nd": "HdBoard",
  "MasVnrType": null,
  "MasVnrArea": 0.0,
  "ExterQual": "TA",
  "ExterCond": "TA",
  "Foundation": "CBlock",
  "BsmtQual": "TA",
  "BsmtCond": "TA",
  "BsmtExposure": "No",
  "BsmtFinType1": "GLQ",
  "BsmtFinSF1": 663,
  "BsmtFinType2": "Unf",
  "BsmtFinSF2": 0,
  "BsmtUnfSF": 396,
  "TotalBsmtSF": 1059,
  "Heati

## Connecting with LLM API

In this step, I am connecting my application with an LLM.

- I will send the house details prompt created earlier to the model. The LLM will analyze the structured JSON data and generate a simple explanation about the house.

- I am using Groq API because it provides fast access to open-source LLM models.

In [79]:
# Installing required library for Groq API
!pip install groq python-dotenv -q
print("API Installation Done!!")

API Installation Done!!


## Loading API Key...

- For security reasons, I am not writing my API key directly inside the notebook.

- I am storing it inside a `.env` file and loading it using python-dotenv.

In [80]:
from groq import Groq

client = Groq(api_key=api_key)

print("Groq client connected successfully!")

Groq client connected successfully!


## Calling LLM Model

- Now I am sending my house analysis prompt to the LLM.

- The model will understand the house features and generate a meaningful response like a real estate assistant.

In [81]:
from groq import Groq

# Creating Groq client
client = Groq(api_key=api_key)

# Sending prompt to LLM
response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ],
    temperature=0.7
)

# Extracting model response
llm_output = response.choices[0].message.content

print(llm_output)

Based on the provided house details, here's the analysis:

1. **Overall quality of the house**: The overall quality of the house is rated 6 out of a possible scale, which is slightly above average. The overall condition is rated 8, indicating that the house is well-maintained.

2. **Important features**: Some important features of the house include:
* 1-story building with a hip roof and composite shingle roofing material
* 3 bedrooms, 1 kitchen, and 1 full bathroom above ground, with 1 half bathroom in the basement
* Attached garage with 1 car capacity and a finished interior
* Central air conditioning and gas heating
* Wooden deck with 192 square feet of space
* Paved driveway and a private fence

3. **Possible buyer advantages**: Potential buyers may appreciate the following advantages:
* The house has been remodeled and updated in 2003, which may have improved its overall condition and functionality
* The presence of central air conditioning and gas heating provides a comfortable l

## Creating Structured Output Format

The previous LLM response was generated as normal text.

- In real applications, it is better to get responses in a structured format like JSON. This makes the output easier to store, validate, and use in other applications.

Here, I am defining a JSON structure for the house analysis response.

In [82]:
# Creating a JSON schema format for expected LLM response

house_analysis_schema = {
    "overall_quality": "",
    "important_features": [],
    "buyer_advantages": [],
    "summary": ""
}

# Displaying expected structure
print(json.dumps(house_analysis_schema, indent=2))

{
  "overall_quality": "",
  "important_features": [],
  "buyer_advantages": [],
  "summary": ""
}


## Creating New Prompt for JSON Response

- Now I am updating my prompt and asking the LLM to return only JSON output.

- This helps maintain a consistent response format and avoids unnecessary text.

In [83]:
# Creating prompt for structured JSON output

structured_prompt = f"""
You are a real estate assistant.

Analyze the following house details.

Return your answer ONLY in JSON format using this structure:

{json.dumps(house_analysis_schema, indent=2)}

House Details:

{json.dumps(house_data, indent=2)}
"""

print(structured_prompt)


You are a real estate assistant.

Analyze the following house details.

Return your answer ONLY in JSON format using this structure:

{
  "overall_quality": "",
  "important_features": [],
  "buyer_advantages": [],
  "summary": ""
}

House Details:

{
  "Id": 893,
  "MSSubClass": 20,
  "MSZoning": "RL",
  "LotFrontage": 70.0,
  "LotArea": 8414,
  "Street": "Pave",
  "Alley": null,
  "LotShape": "Reg",
  "LandContour": "Lvl",
  "Utilities": "AllPub",
  "LotConfig": "Inside",
  "LandSlope": "Gtl",
  "Neighborhood": "Sawyer",
  "Condition1": "Norm",
  "Condition2": "Norm",
  "BldgType": "1Fam",
  "HouseStyle": "1Story",
  "OverallQual": 6,
  "OverallCond": 8,
  "YearBuilt": 1963,
  "YearRemodAdd": 2003,
  "RoofStyle": "Hip",
  "RoofMatl": "CompShg",
  "Exterior1st": "HdBoard",
  "Exterior2nd": "HdBoard",
  "MasVnrType": null,
  "MasVnrArea": 0.0,
  "ExterQual": "TA",
  "ExterCond": "TA",
  "Foundation": "CBlock",
  "BsmtQual": "TA",
  "BsmtCond": "TA",
  "BsmtExposure": "No",
  "BsmtFinT

## Getting Structured Response from LLM

- Now I am sending the updated prompt to the LLM and requesting a JSON formatted response.

In [84]:
# Calling LLM with JSON response instruction

response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {
            "role": "user",
            "content": structured_prompt
        }
    ],
    temperature=0
)

# Extracting response
structured_output = response.choices[0].message.content

print(structured_output)

```
{
  "overall_quality": "Above Average",
  "important_features": [
    "Single-family home",
    "1-story house with hip roof",
    "Attached garage",
    "Wood deck",
    "Private fence"
  ],
  "buyer_advantages": [
    "Well-maintained property with recent remodeling",
    "Good neighborhood location",
    "All public utilities available",
    "Paved street and driveway",
    "Central air conditioning and gas heating"
  ],
  "summary": "This single-family home is a well-maintained, 1-story property with a hip roof and attached garage, located in a good neighborhood with all public utilities available. The house has a wood deck and private fence, and features central air conditioning and gas heating. With a recent remodeling in 2003, this property is a great option for buyers looking for a comfortable and convenient home."
}
```


## Validating LLM Response Using Pydantic

LLM responses can sometimes have missing fields or incorrect formats.

> To make the output reliable, I am using Pydantic to validate the generated JSON response.

It checks:
- Required fields are present
- Lists contain correct values
- Data follows the expected structure

In [85]:
# Installing pydantic library if required later than
!pip install pydantic -q

## Creating Response Model...

Here I am creating a Pydantic model that defines the expected format of the LLM response.

The model contains:
- Overall quality as text
- Important features as a list
- Buyer advantages as a list
- Summary as text

In [86]:
from pydantic import BaseModel
from typing import List
import json

# Defining expected LLM response structure
class HouseAnalysis(BaseModel):
    overall_quality: str
    important_features: List[str]
    buyer_advantages: List[str]
    summary: str

print("Pydantic model created successfully")

Pydantic model created successfully


## Parsing and Validating LLM Output

> The LLM response comes as a string, so first I convert it into a Python dictionary.

- After that, I pass it through the Pydantic model for validation.

In [87]:
# Removing markdown code block if LLM added it
clean_json = structured_output.replace("```json", "").replace("```", "").strip()

# Converting JSON string into Python dictionary
output_dict = json.loads(clean_json)


# Validating response using Pydantic
validated_response = HouseAnalysis(**output_dict)


print(validated_response)

overall_quality='Above Average' important_features=['Single-family home', '1-story house with hip roof', 'Attached garage', 'Wood deck', 'Private fence'] buyer_advantages=['Well-maintained property with recent remodeling', 'Good neighborhood location', 'All public utilities available', 'Paved street and driveway', 'Central air conditioning and gas heating'] summary='This single-family home is a well-maintained, 1-story property with a hip roof and attached garage, located in a good neighborhood with all public utilities available. The house has a wood deck and private fence, and features central air conditioning and gas heating. With a recent remodeling in 2003, this property is a great option for buyers looking for a comfortable and convenient home.'


## Final Result

The complete workflow is now:

1. Loaded house dataset
2. Selected sample house records
3. Converted records into JSON
4. Created an LLM prompt
5. Generated house analysis using LLM
6. Requested structured JSON output
7. Validated LLM response using Pydantic

This makes the LLM application more reliable and easier to integrate with other systems.

## Exporting Validated LLM Response

- After validating the LLM response, I am converting the output into a clean JSON format.

- Saving the response is useful because it can be reused later by another application or stored as an API response.

In [88]:
# Converting validated Pydantic object into dictionary
final_output = validated_response.model_dump()

# Displaying final JSON output
print(json.dumps(final_output, indent=2))

{
  "overall_quality": "Above Average",
  "important_features": [
    "Single-family home",
    "1-story house with hip roof",
    "Attached garage",
    "Wood deck",
    "Private fence"
  ],
  "buyer_advantages": [
    "Well-maintained property with recent remodeling",
    "Good neighborhood location",
    "All public utilities available",
    "Paved street and driveway",
    "Central air conditioning and gas heating"
  ],
  "summary": "This single-family home is a well-maintained, 1-story property with a hip roof and attached garage, located in a good neighborhood with all public utilities available. The house has a wood deck and private fence, and features central air conditioning and gas heating. With a recent remodeling in 2003, this property is a great option for buyers looking for a comfortable and convenient home."
}


## Saving Output as JSON File...

- Now I am saving the final validated response into a JSON file.

- This file can be used later for storing LLM generated analysis results.

In [89]:
# Saving final output into JSON file
with open("house_analysis_output.json", "w") as file:
    json.dump(final_output, file, indent=4)

print("File saved successfully!!")

File saved successfully!!


## Checking Saved File

> I am loading the saved JSON file again to confirm that the data was stored correctly.

In [90]:
# Reading saved JSON file

with open("house_analysis_output.json", "r") as file:
    saved_data = json.load(file)

print(json.dumps(saved_data, indent=2))

{
  "overall_quality": "Above Average",
  "important_features": [
    "Single-family home",
    "1-story house with hip roof",
    "Attached garage",
    "Wood deck",
    "Private fence"
  ],
  "buyer_advantages": [
    "Well-maintained property with recent remodeling",
    "Good neighborhood location",
    "All public utilities available",
    "Paved street and driveway",
    "Central air conditioning and gas heating"
  ],
  "summary": "This single-family home is a well-maintained, 1-story property with a hip roof and attached garage, located in a good neighborhood with all public utilities available. The house has a wood deck and private fence, and features central air conditioning and gas heating. With a recent remodeling in 2003, this property is a great option for buyers looking for a comfortable and convenient home."
}


## Creating Streamlit Application

- In this step, I am creating a simple user interface for my LLM-powered house analysis system.

- The user can select a house record from the dataset, and the application will generate an AI-based property analysis using the LLM model.

This makes my project easier to use without running notebook cells manually.

In [91]:
# Installing Streamlit library
!pip install streamlit -q

In [92]:
import streamlit as st
import pandas as pd
import json
from groq import Groq

# Loading dataset
df = pd.read_csv("cleaned_data.csv")

# Streamlit page settings
st.title("🏠 AI House Analysis Assistant")

st.write(
    "Select a house record and get an AI-generated property analysis."
)

# Selecting house ID
house_id = st.selectbox(
    "Select House ID",
    df["Id"].tolist()
)

# Getting selected house data
selected_house = df[df["Id"] == house_id].iloc[0]

# Converting dataframe row into dictionary
house_data = selected_house.where(
    pd.notnull(selected_house),
    None
).to_dict()

# Button for generating analysis
if st.button("Analyze House"):

    prompt = f"""
    You are a real estate assistant.

    Analyze this house information.

    Return ONLY JSON format:

    {{
      "overall_quality": "",
      "important_features": [],
      "buyer_advantages": [],
      "summary": ""
    }}

    House Details:
    {json.dumps(house_data, indent=2)}
    """

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0
    )

    output = response.choices[0].message.content

    # Cleaning JSON response
    output = output.replace("```json", "").replace("```", "").strip()

    result = json.loads(output)

    st.subheader("AI Analysis")

    st.write("### Overall Quality")
    st.write(result["overall_quality"])

    st.write("### Important Features")
    for feature in result["important_features"]:
        st.write("✔️", feature)

    st.write("### Buyer Advantages")
    for advantage in result["buyer_advantages"]:
        st.write("✔️", advantage)

    st.write("### Summary")
    st.write(result["summary"])

2026-07-13 17:39:49.096 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-13 17:39:49.097 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-13 17:39:49.101 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-13 17:39:49.103 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-13 17:39:49.108 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-13 17:39:49.110 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-13 17:39:49.113 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-13 17:39:49.124 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

## Running the Streamlit Application

- After creating the `app.py` file, I ran the following command to start the Streamlit application:

```bash
streamlit run app.py
```

- Since I developed this project in Google Colab, I exposed the local Streamlit server using ngrok so I could access the application in my browser.

In [99]:
%%writefile app.py


Overwriting app.py


In [ ]:
import streamlit as st
import pandas as pd
import json
from groq import Groq

# Loading dataset
df = pd.read_csv("cleaned_data.csv")

# Streamlit page settings
st.title("🏠 AI House Analysis Assistant")

st.write(
    "Select a house record and get an AI-generated property analysis."
)

# Selecting house ID
house_id = st.selectbox(
    "Select House ID",
    df["Id"].tolist()
)

# Getting selected house data
selected_house = df[df["Id"] == house_id].iloc[0]

# Converting dataframe row into dictionary
house_data = selected_house.where(
    pd.notnull(selected_house),
    None
).to_dict()

# Button for generating analysis
if st.button("Analyze House"):

    prompt = f"""
    You are a real estate assistant.

    Analyze this house information.

    Return ONLY JSON format:

    {{
      "overall_quality": "",
      "important_features": [],
      "buyer_advantages": [],
      "summary": ""
    }}

    House Details:
    {json.dumps(house_data, indent=2)}
    """

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0
    )

    output = response.choices[0].message.content

    # Cleaning JSON response
    output = output.replace("```json", "").replace("```", "").strip()

    result = json.loads(output)

    st.subheader("AI Analysis")

    st.write("### Overall Quality")
    st.write(result["overall_quality"])

    st.write("### Important Features")
    for feature in result["important_features"]:
        st.write("✔️", feature)

    st.write("### Buyer Advantages")
    for advantage in result["buyer_advantages"]:
        st.write("✔️", advantage)

    st.write("### Summary")
    st.write(result["summary"])

2026-07-13 17:39:49.096 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-13 17:39:49.097 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-13 17:39:49.101 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-13 17:39:49.103 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-13 17:39:49.108 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-13 17:39:49.110 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-13 17:39:49.113 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-13 17:39:49.124 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar